In [4]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
# ИСПРАВЛЕНИЕ ИМПОРТА: Импортируем модуль MACCS напрямую
from rdkit.Chem import MACCSkeys 
import re

# Загружаем наш очищенный датасет
INPUT_FILE = "dataset_clean.csv"
OUTPUT_FILE = "dataset_with_all_descriptors.csv"

df = pd.read_csv(INPUT_FILE)
print(f"Загружено строк: {len(df)}")

# Инициализируем списки под новые признаки
features = {
    # Фаза 2: Классические дескрипторы
    'mw': [], 'logp': [], 'tpsa': [], 'hba': [], 'hbd': [], 'rot_bonds': [], 'aromatic_rings': [],
    # Фаза 4: Metal-Aware дескрипторы
    'is_Pt': [], 'is_Ru': [], 'is_Rh': [], 'is_Ir': [], 'is_Au': [], 'is_Pd': [],
    'has_halogen': [], 'has_sulfur': [], 'charge': []
}

# Списки для фингерпринтов (Фаза 3)
morgan_bits = []
maccs_bits = []

print("Начинаем расчет дескрипторов...")
for idx, row in df.iterrows():
    smiles = row['canonical_smiles']
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        # Если вдруг закралась совсем нечитаемая структура
        for k in features.keys(): features[k].append(np.nan)
        morgan_bits.append([np.nan]*256)
        maccs_bits.append([np.nan]*167)
        continue
        
    # --- ФАЗА 2: Классические свойства ---
    features['mw'].append(Descriptors.MolWt(mol))
    features['logp'].append(Descriptors.MolLogP(mol))
    features['tpsa'].append(Descriptors.TPSA(mol))
    features['hba'].append(Descriptors.NumHAcceptors(mol))
    features['hbd'].append(Descriptors.NumHDonors(mol))
    features['rot_bonds'].append(Descriptors.NumRotatableBonds(mol))
    features['aromatic_rings'].append(Descriptors.NumAromaticRings(mol))
    
    # --- ФАЗА 4: Metal-Aware (Парсинг структуры) ---
    smiles_lower = smiles.lower()
    features['is_Pt'].append(1 if '[pt' in smiles_lower else 0)
    features['is_Ru'].append(1 if '[ru' in smiles_lower else 0)
    features['is_Rh'].append(1 if '[rh' in smiles_lower else 0)
    features['is_Ir'].append(1 if '[ir' in smiles_lower else 0)
    features['is_Au'].append(1 if '[au' in smiles_lower else 0)
    features['is_Pd'].append(1 if '[pd' in smiles_lower else 0)
    
    # Наличие уходящих групп (галогены) или координация по сере
    features['has_halogen'].append(1 if any(x in smiles_lower for x in ['cl', 'br', 'i']) else 0)
    features['has_sulfur'].append(1 if 's' in smiles_lower else 0)
    
    # Чистый формальный заряд всей молекулы
    features['charge'].append(Chem.GetFormalCharge(mol))
    
    # --- ФАЗА 3: Фингерпринты ---
    # Morgan (радиус 2, 256 бит)
    fp_morgan = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=256)
    morgan_bits.append(list(fp_morgan))
    
    # ИСПРАВЛЕНО: Безопасный вызов MACCS через отдельный модуль
    fp_maccs = MACCSkeys.GenMACCSKeys(mol)
    maccs_bits.append(list(fp_maccs))

# Добавляем рассчитанные признаки в основной датафрейм
for k, values in features.items():
    df[k] = values

# Формируем датафреймы для фингерпринтов
df_morgan = pd.DataFrame(morgan_bits, columns=[f'morgan_{i}' for i in range(256)])
df_maccs = pd.DataFrame(maccs_bits, columns=[f'maccs_{i}' for i in range(167)])

# Объединяем геометрию и фингерпринты
df_all = pd.concat([df.reset_index(drop=True), df_morgan, df_maccs], axis=1)

# --- УМНЫЙ ХОД ДЛЯ КЛЕТОЧНЫХ ЛИНИЙ: One-Hot Encoding в числовом формате (0 или 1) ---
df_all = pd.get_dummies(df_all, columns=['target_name'], prefix='cell_line', dtype=int)

# На всякий случай удаляем строки, где расчет не удался
df_all = df_all.dropna(subset=['mw'])

# Сохраняем мега-датасет
df_all.to_csv(OUTPUT_FILE, index=False)

print("\n Мега-датасет успешно собран!")
print(f"Итоговая размерность таблицы: {df_all.shape} (строки, признаки)")
print(f"Файл сохранен как: {OUTPUT_FILE}")

Загружено строк: 142
Начинаем расчет дескрипторов...


[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerator
[11:43:56] DEPRECATION WARNING: please use MorganGenerat


 Мега-датасет успешно собран!
Итоговая размерность таблицы: (142, 448) (строки, признаки)
Файл сохранен как: dataset_with_all_descriptors.csv


[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator
[11:43:57] DEPRECATION WARNING: please use MorganGenerator


In [6]:
df_all.head(3)

,chembl_id,chemical_formula,pIC50,canonical_smiles,pref_name,mw,logp,tpsa,hba,hbd,...,maccs_161,maccs_162,maccs_163,maccs_164,maccs_165,maccs_166,cell_line_41M,cell_line_A2780,cell_line_CH1,cell_line_SK-OV-3
0,CHEMBL102583,C7H18Cl2N2O2Pt,5.720,[Pt](N)(Cl)(Cl)(O)(O)CN1CCCCC1C,"trans-a-Ammine-b,d-dichloro-c,e-dihydroxo-f-(N...",428.217,1.0395,69.72,4,3,...,1,0,1,1,1,0,1,0,0,0
1,CHEMBL102583,C7H18Cl2N2O2Pt,5.595,[Pt](N)(Cl)(Cl)(O)(O)CN1CCCCC1C,"trans-a-Ammine-b,d-dichloro-c,e-dihydroxo-f-(N...",428.217,1.0395,69.72,4,3,...,1,0,1,1,1,0,0,1,0,0
2,CHEMBL102583,C7H18Cl2N2O2Pt,5.715,[Pt](N)(Cl)(Cl)(O)(O)CN1CCCCC1C,"trans-a-Ammine-b,d-dichloro-c,e-dihydroxo-f-(N...",428.217,1.0395,69.72,4,3,...,1,0,1,1,1,0,0,0,1,0


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# 1. ЗАГРУЗКА ДАННЫХ
df = pd.read_csv("dataset_with_all_descriptors.csv")
y = df['pIC50']
groups = df['chembl_id']

# Определение групп признаков
classical_cols = ['mw', 'logp', 'tpsa', 'hba', 'hbd', 'rot_bonds', 'aromatic_rings']
cell_line_cols = [c for c in df.columns if 'cell_line_' in c]
fingerprint_cols = [c for c in df.columns if 'morgan_' in c or 'maccs_' in c]
metal_cols = ['is_Pt', 'is_Ru', 'is_Rh', 'is_Ir', 'is_Au', 'is_Pd', 'has_halogen', 'has_sulfur', 'charge']

# Наборы фичей для каждой Фазы
phases = {
    "PHASE 2 (Classical Baseline)": classical_cols + cell_line_cols,
    "PHASE 3 (+ Fingerprints)": classical_cols + cell_line_cols + fingerprint_cols,
    "PHASE 4 (+ Metal-Aware Features)": classical_cols + cell_line_cols + fingerprint_cols + metal_cols
}

# 2. ОПРЕДЕЛЕНИЕ МОДЕЛЕЙ
def get_models():
    models = {
        "Linear Regression": LinearRegression(),
        "Ridge (Regularized LR)": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    }
    if xgb: models["XGBoost"] = xgb.XGBRegressor(n_estimators=100, random_state=42, max_depth=4, learning_rate=0.05)
    if lgb: models["LightGBM"] = lgb.LGBMRegressor(n_estimators=100, random_state=42, max_depth=4, verbose=-1)
    if cb:  models["CatBoost"] = cb.CatBoostRegressor(n_estimators=150, random_state=42, depth=4, verbose=0)
    return models

# 3. ЦИКЛ КРОСС-ВАЛИДАЦИИ И ИССЛЕДОВАНИЯ
results = []

print("=== ЗАПУСК ПАЙПЛАЙНА QSАR С ГРУППОВОЙ ВАЛИДАЦИЕЙ ===")
gkf = GroupKFold(n_splits=5)

for phase_name, feature_list in phases.items():
    print(f"\n Оцениваем {phase_name}... (Признаков: {len(feature_list)})")
    X = df[feature_list]
    
    models = get_models()
    for model_name, model in models.items():
        cv_r2, cv_rmse = [], []
        
        for train_idx, test_idx in gkf.split(X, y, groups):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            # Обучение и предикт
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            
            cv_r2.append(r2_score(y_test, preds))
            cv_rmse.append(root_mean_squared_error(y_test, preds))
            
        mean_r2 = np.mean(cv_r2)
        mean_rmse = np.mean(cv_rmse)
        
        results.append({
            "Phase": phase_name,
            "Model": model_name,
            "CV R²": mean_r2,
            "CV RMSE": mean_rmse
        })
        print(f"  -> {model_name:25} | CV R²: {mean_r2:6.3f} | CV RMSE: {mean_rmse:.3f}")

# Сводная таблица результатов
df_res = pd.DataFrame(results)
print("\n Пайплайн завершен!")

=== ЗАПУСК ПАЙПЛАЙНА QSАR С ГРУППОВОЙ ВАЛИДАЦИЕЙ ===

 Оцениваем PHASE 2 (Classical Baseline)... (Признаков: 11)
  -> Linear Regression         | CV R²:  0.165 | CV RMSE: 0.531
  -> Ridge (Regularized LR)    | CV R²:  0.167 | CV RMSE: 0.531
  -> Random Forest             | CV R²:  0.089 | CV RMSE: 0.555
  -> XGBoost                   | CV R²: -0.017 | CV RMSE: 0.593
  -> LightGBM                  | CV R²: -0.005 | CV RMSE: 0.588
  -> CatBoost                  | CV R²:  0.109 | CV RMSE: 0.553

 Оцениваем PHASE 3 (+ Fingerprints)... (Признаков: 434)
  -> Linear Regression         | CV R²:  0.357 | CV RMSE: 0.463
  -> Ridge (Regularized LR)    | CV R²:  0.362 | CV RMSE: 0.462
  -> Random Forest             | CV R²:  0.267 | CV RMSE: 0.500
  -> XGBoost                   | CV R²:  0.224 | CV RMSE: 0.504
  -> LightGBM                  | CV R²:  0.303 | CV RMSE: 0.489
  -> CatBoost                  | CV R²:  0.347 | CV RMSE: 0.468

 Оцениваем PHASE 4 (+ Metal-Aware Features)... (Признаков: 44

Модели, обученные исключительно на физико-химических константах (молекулярный вес, LogP, TPSA) и биологическом контексте (клеточные линии), показали крайне низкую прогностическую способность (Rmax2=0.167 для Ridge). Тяжелые древесные ансамбли (XGBoost, LightGBM) продемонстрировали отрицательные значения R2, что указывает на переобучение. Физико-химические константы верхнего уровня не способны отразить тонкие различия в координационных сферах изоструктурных комплексов.

Добавление структурных фингерпринтов вызвало качественный скачок точности для всех алгоритмов (R2 вырос до 0.35−0.36). Это наглядно доказывает, что ключевым фактором, определяющим противоопухолевую активность in vitro, является специфическое строение и пространственное окружение органических лигандов.

Интеграция металл-специфичных признаков (тип металла, общий формальный заряд, наличие галогеновых уходящих групп) позволила достичь наилучших метрик. Абсолютным лидером стал алгоритм CatBoost, показавший CV R2=0.379 при ошибке RMSE=0.457

Близость метрик линейной Ridge-регрессии (R2=0.366) и нелинейного CatBoost (R2=0.379) в сочетании с небольшим объемом выборки (142 строки) указывает на то, что вклад дескрипторов лигандов носит преимущественно аддитивный характер, а усложнение архитектуры моделей сдерживается «проклятием размерности». Ошибка RMSE≈0.45 логарифмических единиц является хорошим академическим результатом, сопоставимым с экспериментальной погрешностью биологических тестов in vitro.

Полученные результаты указывают на то, что разработанная на данном этапе QSAR-модель не может рассматриваться как готовый к внедрению лабораторный инструмент для прямой прогностической оценки активности in vitro новых металлокомплексов. Вместо этого в рамках проекта сформирован высокоэффективный прототип и методологический пайплайн, который наглядно демонстрирует правильные векторы развития: точность предсказаний строго лимитирована текущим объемом выборки ("проклятием размерности") и встроенным биологическим шумом, однако разработанная архитектура способна масштабироваться и выйти на прикладной уровень предсказательной силы по мере естественного накопления и добавления новых экспериментальных данных